# ReceiptGuard — Fresh Colab Tester

This notebook provides the standalone core of ReceiptGuard without the app interface.

It includes:

1. A hidden Gemini API-key prompt
2. One required function: `check_receipt(image)`
3. Python checks for arithmetic, GST, ABN, dates and duplicates
4. One-receipt testing
5. Batch testing and automatic CSV download

**Important:** The result is a screening indicator, not proof that a receipt is fraudulent.

Run each numbered step separately. Do **not** press **Run all** the first time.


## Step 1 — Install the small set of required packages

Press the play button once and wait for **Step 1 complete**.


In [1]:
import requests
import pandas as pd
print("Step 1 complete. Required packages are already available in Colab.")


Step 1 complete. Required packages are already available in Colab.


## Step 2 — Enter your Gemini API key privately

The key is hidden while you type or paste it. It is kept only in the current Colab session and is not written to the CSV.


In [2]:
from getpass import getpass

GEMINI_API_KEY = getpass("Paste Gemini API key (hidden), then press Enter: ").strip()
if not GEMINI_API_KEY:
    raise ValueError("No API key was entered.")

MODEL_NAMES = ["gemini-2.5-flash", "gemini-3.6-flash", "gemini-3.8-flash"]
print("Step 2 complete. API key accepted and kept hidden.")
print("Automatic fallback models ready:", ", ".join(MODEL_NAMES))


Paste Gemini API key (hidden), then press Enter: ··········
Step 2 complete. API key accepted and kept hidden.
Automatic fallback models ready: gemini-2.5-flash, gemini-3.6-flash, gemini-3.8-flash


## Step 3 — Create `check_receipt(image)`

This cell contains the Gemini extraction prompt and the Python verification rules. Run it once and wait for **Step 3 complete**.


In [3]:
import base64
import json
import mimetypes
import os
import re
import time
from datetime import datetime, date
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

SEEN_SIGNATURES = set()

EXTRACTION_PROMPT = r"""
You are the extraction component of ReceiptGuard, an Australian receipt and tax-invoice screening tool.

Read the attached receipt or invoice carefully. Return ONLY valid JSON. Do not use markdown fences.
Use null when a field is not visible. Do not invent missing values.

Required JSON structure:
{
  "merchant_name": "string or null",
  "abn": "digits/string or null",
  "date": "YYYY-MM-DD or null",
  "currency": "AUD or another currency or null",
  "line_items": [
    {"description": "string", "quantity": 1, "unit_price": 0.0, "amount": 0.0}
  ],
  "subtotal": 0.0,
  "gst": 0.0,
  "total": 0.0,
  "confidence": "high, medium, or low",
  "extraction_notes": ["short note if anything is unclear"]
}

Important instructions:
- Amounts must be numbers without currency symbols.
- Keep printed values exactly as shown; do not silently correct arithmetic.
- If a line item has no separate quantity or unit price, use null for those fields.
- This task is extraction only. Do not decide whether the document is fake.
"""

def _number(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)):
        return float(value)
    cleaned = re.sub(r"[^0-9.\-]", "", str(value))
    try:
        return float(cleaned) if cleaned not in {"", "-", "."} else None
    except ValueError:
        return None

def _valid_abn(value):
    if not value:
        return None
    digits = re.sub(r"\D", "", str(value))
    if len(digits) != 11:
        return False
    numbers = [int(x) for x in digits]
    numbers[0] -= 1
    weights = [10, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
    return sum(n * w for n, w in zip(numbers, weights)) % 89 == 0

def _parse_json(text):
    text = text.strip()
    fence = chr(96) * 3
    text = text.replace(fence + "json", "").replace(fence, "").strip()
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("Gemini did not return a JSON object.")
    return json.loads(text[start:end + 1])

def _gemini_extract(image):
    if isinstance(image, (str, os.PathLike)):
        path = Path(image)
        raw = path.read_bytes()
        mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    elif isinstance(image, bytes):
        raw = image
        mime = "image/jpeg"
    else:
        raise TypeError("image must be a filename/path or image bytes")

    payload = {
        "contents": [{
            "role": "user",
            "parts": [
                {"text": EXTRACTION_PROMPT},
                {"inline_data": {"mime_type": mime, "data": base64.b64encode(raw).decode("ascii")}},
            ],
        }],
        "generationConfig": {
            "temperature": 0,
            "responseMimeType": "application/json",
        },
    }

    last_error = None
    tried = []
    for model_name in MODEL_NAMES:
        tried.append(model_name)
        endpoint = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
        for attempt in range(2):
            try:
                response = requests.post(
                    endpoint,
                    headers={"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"},
                    json=payload,
                    timeout=60,
                )
                if response.status_code in {404}:
                    last_error = RuntimeError(f"Model {model_name} is unavailable for this key")
                    break
                if response.status_code in {429, 500, 502, 503, 504}:
                    raise RuntimeError(f"Temporary error {response.status_code} on {model_name}")
                response.raise_for_status()
                parts = response.json()["candidates"][0]["content"]["parts"]
                answer = "".join(p.get("text", "") for p in parts)
                receipt = _parse_json(answer)
                receipt["model_used"] = model_name
                return receipt
            except Exception as exc:
                last_error = exc
                if attempt == 0:
                    time.sleep(2)
        print(f"Switching from {model_name} to the next fallback model...")
    raise RuntimeError(f"All Gemini fallback models failed ({', '.join(tried)}). Last error: {last_error}")

def _run_python_checks(receipt):
    checks = []
    score = 0

    def add(name, status, detail, points=0):
        nonlocal score
        checks.append({"check": name, "status": status, "detail": detail})
        if status == "FLAG":
            score += points

    items = receipt.get("line_items") or []
    amounts = [_number(item.get("amount")) for item in items if isinstance(item, dict)]
    amounts = [x for x in amounts if x is not None]
    subtotal = _number(receipt.get("subtotal"))
    gst = _number(receipt.get("gst"))
    total = _number(receipt.get("total"))

    if amounts and subtotal is not None:
        calculated = round(sum(amounts), 2)
        difference = abs(calculated - subtotal)
        add(
            "Line items vs subtotal",
            "PASS" if difference <= 0.10 else "FLAG",
            f"Items add to ${calculated:.2f}; printed subtotal is ${subtotal:.2f}.",
            2,
        )
    else:
        add("Line items vs subtotal", "UNABLE", "Not enough readable line-item information.")

    if subtotal is not None and gst is not None and total is not None:
        calculated = round(subtotal + gst, 2)
        difference = abs(calculated - total)
        add(
            "Subtotal plus GST vs total",
            "PASS" if difference <= 0.10 else "FLAG",
            f"Subtotal + GST is ${calculated:.2f}; printed total is ${total:.2f}.",
            2,
        )
    else:
        add("Subtotal plus GST vs total", "UNABLE", "Subtotal, GST or total is missing.")

    if gst is not None and total not in {None, 0}:
        common_gst = round(total / 11, 2)
        difference = abs(gst - common_gst)
        add(
            "GST reasonableness",
            "PASS" if difference <= 0.20 else "FLAG",
            f"Printed GST is ${gst:.2f}; 1/11 of total is ${common_gst:.2f}. Mixed/GST-free sales may explain a difference.",
            1,
        )
    else:
        add("GST reasonableness", "UNABLE", "GST or total is missing.")

    abn = receipt.get("abn")
    abn_result = _valid_abn(abn)
    if abn_result is True:
        add("ABN checksum", "PASS", "The eleven-digit ABN checksum is valid.")
    elif abn_result is False:
        add("ABN checksum", "FLAG", "The printed ABN fails the official checksum format.", 2)
    else:
        add("ABN checksum", "UNABLE", "No readable ABN was found.")

    receipt_date = receipt.get("date")
    if receipt_date:
        try:
            parsed = datetime.strptime(str(receipt_date), "%Y-%m-%d").date()
            if parsed > date.today():
                add("Date logic", "FLAG", f"Receipt date {parsed} is in the future.", 1)
            else:
                add("Date logic", "PASS", f"Receipt date {parsed} is not in the future.")
        except ValueError:
            add("Date logic", "UNABLE", "The date could not be interpreted as YYYY-MM-DD.")
    else:
        add("Date logic", "UNABLE", "No readable date was found.")

    merchant = str(receipt.get("merchant_name") or "").strip().lower()
    signature = (merchant, str(receipt_date or ""), round(total, 2) if total is not None else None)
    if merchant and receipt_date and total is not None:
        if signature in SEEN_SIGNATURES:
            add("Possible duplicate", "FLAG", "The same merchant, date and total were already checked in this session.", 2)
        else:
            add("Possible duplicate", "PASS", "No matching merchant/date/total was previously checked in this session.")
            SEEN_SIGNATURES.add(signature)
    else:
        add("Possible duplicate", "UNABLE", "Merchant, date or total is missing.")

    confidence = str(receipt.get("confidence") or "low").lower()
    if confidence == "low":
        score += 1

    if score >= 4:
        classification, risk = "SUSPICIOUS", "HIGH"
        advice = "Request the bank/card statement and manually verify the merchant before reimbursement."
    elif score >= 2:
        classification, risk = "REVIEW NEEDED", "MEDIUM"
        advice = "A person should review the flagged fields and supporting evidence."
    else:
        classification, risk = "LIKELY GENUINE", "LOW"
        advice = "No strong rule-based inconsistency was found, but this is not proof of authenticity."

    return checks, score, classification, risk, advice

def check_receipt(image):
    """Extract one receipt with Gemini, run Python checks, and return a result dictionary."""
    filename = Path(image).name if isinstance(image, (str, os.PathLike)) else "uploaded_image"
    try:
        receipt = _gemini_extract(image)
        checks, score, classification, risk, advice = _run_python_checks(receipt)
        return {
            "filename": filename,
            "classification": classification,
            "risk_rating": risk,
            "risk_score": score,
            "merchant_name": receipt.get("merchant_name"),
            "date": receipt.get("date"),
            "abn": receipt.get("abn"),
            "subtotal": receipt.get("subtotal"),
            "gst": receipt.get("gst"),
            "total": receipt.get("total"),
            "confidence": receipt.get("confidence"),
            "model_used": receipt.get("model_used"),
            "checks": checks,
            "advice": advice,
            "error": "",
        }
    except Exception as exc:
        return {
            "filename": filename,
            "classification": "INCONCLUSIVE",
            "risk_rating": "UNKNOWN",
            "risk_score": "",
            "merchant_name": "",
            "date": "",
            "abn": "",
            "subtotal": "",
            "gst": "",
            "total": "",
            "confidence": "low",
            "model_used": "",
            "checks": [],
            "advice": "Try a clearer image or retry later. Do not treat an error as a fraud result.",
            "error": str(exc),
        }

def show_result(result):
    print("\nClassification:", result["classification"])
    print("Risk rating:", result["risk_rating"])
    print("Merchant:", result["merchant_name"])
    print("Total:", result["total"])
    if result["error"]:
        print("Error:", result["error"])
    for check in result["checks"]:
        print(f'- {check["status"]}: {check["check"]} — {check["detail"]}')
    print("Advice:", result["advice"])

print("Step 3 complete. Function ready: check_receipt(image)")


Step 3 complete. Function ready: check_receipt(image)


## Step 4 — Test one receipt first

Press play, click **Choose Files**, and select one JPG, JPEG, PNG, WEBP or PDF receipt. The result should normally appear within 90 seconds.


In [4]:
from google.colab import files

one_upload = files.upload()
if not one_upload:
    print("No file was selected.")
else:
    one_filename = next(iter(one_upload))
    print("Checking:", one_filename)
    one_result = check_receipt(one_filename)
    show_result(one_result)


Saving 21. REDDY EXPRESS RECIEPT.jpeg to 21. REDDY EXPRESS RECIEPT.jpeg
Checking: 21. REDDY EXPRESS RECIEPT.jpeg
Switching from gemini-2.5-flash to the next fallback model...

Classification: REVIEW NEEDED
Risk rating: MEDIUM
Merchant: Reddy Express Doncaster
Total: 39.68
- PASS: Line items vs subtotal — Items add to $39.68; printed subtotal is $39.68.
- FLAG: Subtotal plus GST vs total — Subtotal + GST is $43.29; printed total is $39.68.
- PASS: GST reasonableness — Printed GST is $3.61; 1/11 of total is $3.61. Mixed/GST-free sales may explain a difference.
- PASS: ABN checksum — The eleven-digit ABN checksum is valid.
- PASS: Date logic — Receipt date 2026-10-04 is not in the future.
- PASS: Possible duplicate — No matching merchant/date/total was previously checked in this session.
Advice: A person should review the flagged fields and supporting evidence.


## Step 5 — Batch test receipts and download a CSV

Select several receipt images at the same time. For easier evaluation, begin genuine filenames with `real_` and generated filenames with `fake_`, for example:

- `real_01.jpg`
- `fake_01.png`

The notebook will run every receipt through `check_receipt(image)`, display the table and download `ReceiptGuard_results.csv`.


In [23]:
from google.colab import files

batch_upload = files.upload()
supported = (".jpg", ".jpeg", ".png", ".webp", ".pdf")
filenames = [name for name in batch_upload if name.lower().endswith(supported)]
SEEN_SIGNATURES.clear()  # Start a fresh duplicate-check history for this batch.

if not filenames:
    print("No supported receipt files were selected.")
else:
    rows = []
    for index, filename in enumerate(filenames, start=1):
        print(f"Checking {index}/{len(filenames)}: {filename}")
        result = check_receipt(filename)
        lower_name = filename.lower()
        expected = "Real" if lower_name.startswith(("real_", "genuine_")) else (
            "Fake" if lower_name.startswith(("fake_", "generated_")) else ""
        )
        if expected == "Real":
            outcome = "PASS" if result["classification"] == "LIKELY GENUINE" else "FAIL"
        elif expected == "Fake":
            outcome = "PASS" if result["classification"] in {"SUSPICIOUS", "REVIEW NEEDED"} else "FAIL"
        else:
            outcome = "ADD EXPECTED LABEL"

        rows.append({
            "filename": filename,
            "expected_label": expected,
            "classification": result["classification"],
            "risk_rating": result["risk_rating"],
            "model_used": result["model_used"],
            "outcome": outcome,
            "merchant_name": result["merchant_name"],
            "date": result["date"],
            "abn": result["abn"],
            "subtotal": result["subtotal"],
            "gst": result["gst"],
            "total": result["total"],
            "red_flags": " | ".join(
                c["check"] + ": " + c["detail"]
                for c in result["checks"] if c["status"] == "FLAG"
            ),
            "advice": result["advice"],
            "error": result["error"],
        })

    results_df = pd.DataFrame(rows)
    display(results_df)
    output_name = "/content/ReceiptGuard_results.csv"
    results_df.to_csv(output_name, index=False, encoding="utf-8-sig")
    print("CSV ready. Download starting now.")
    files.download(output_name)


Saving 25. METRO HOME RECIEPT.png to 25. METRO HOME RECIEPT (2).png
Checking 1/1: 25. METRO HOME RECIEPT (2).png
Switching from gemini-2.5-flash to the next fallback model...


,filename,expected_label,classification,risk_rating,model_used,outcome,merchant_name,date,abn,subtotal,gst,total,red_flags,advice,error
0,25. METRO HOME RECIEPT (2).png,,SUSPICIOUS,HIGH,gemini-3.6-flash,ADD EXPECTED LABEL,Metro Home & More,2025-06-25,64 123 987 654,32.0,3.5,32.0,Subtotal plus GST vs total: Subtotal + GST is ...,Request the bank/card statement and manually v...,


CSV ready. Download starting now.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 6 — Calculate your evaluation figures

Run this only after Step 5 and after your filenames contain the expected labels.


In [17]:
labelled = results_df[results_df["expected_label"].isin(["Real", "Fake"])].copy()
fake_rows = labelled[labelled["expected_label"] == "Fake"]
real_rows = labelled[labelled["expected_label"] == "Real"]

detected_fakes = fake_rows["classification"].isin(["SUSPICIOUS", "REVIEW NEEDED"]).sum()
false_alarms = (~real_rows["classification"].eq("LIKELY GENUINE")).sum()

detection_rate = (detected_fakes / len(fake_rows) * 100) if len(fake_rows) else None
false_alarm_rate = (false_alarms / len(real_rows) * 100) if len(real_rows) else None

print(f"Fake receipts detected: {detected_fakes}/{len(fake_rows)}")
print("Detection rate:", f"{detection_rate:.1f}%" if detection_rate is not None else "No fake receipts labelled")
print(f"Real receipts wrongly flagged: {false_alarms}/{len(real_rows)}")
print("False-alarm rate:", f"{false_alarm_rate:.1f}%" if false_alarm_rate is not None else "No real receipts labelled")
print("Remember: inconclusive and error cases should be reported, not hidden.")


Fake receipts detected: 0/0
Detection rate: No fake receipts labelled
Real receipts wrongly flagged: 0/0
False-alarm rate: No real receipts labelled
Remember: inconclusive and error cases should be reported, not hidden.


## What to submit or screenshot

- This notebook or its shareable Colab link
- `ReceiptGuard_results.csv`
- A screenshot showing one genuine result
- A screenshot showing one fake/suspicious result
- Your detection rate, false-alarm rate and any errors

### Method note

Gemini extracts fields from the document. The final screening classification is based on transparent Python rules rather than asking Gemini whether the image “looks fake.” Checks are indicators only; unusual tax treatment, poor image quality or OCR mistakes can create false alarms.

### Technical reference

The notebook calls Google’s official Gemini `models.generateContent` REST endpoint and sends the receipt as inline media. Research starting points: GPT4o-Receipt (arXiv:2603.11442) and AIForge-Doc (arXiv:2602.20569).
